In [ ]:
#@markdown # Google Drive
Mount_Drive = True  #@param {type:"boolean"}
Client_Secrets_Path = ""  #@param {type:"string"}
#@markdown <sub>Optional path inside mounted Drive to `client_secrets.json`.</sub>
FOLDER_ID = "/content/gdrive/MyDrive"  #@param {type:"string"}
#@markdown <sub>Optional Drive folder ID containing your `LoRA` folders. Pre-fills the Manage tab.</sub>

#@markdown ## Web interface
Public_Share = True  #@param {type:"boolean"}
Port = 7871  #@param {type:"integer"}
#@markdown ---

import os, shutil, subprocess, sys, time
from pathlib import Path

REPO_URL = "https://github.com/antopou/prompt-generator.git"
REPO_DIR = "prompt-generator"
TOTAL = 5

# ---- log helpers (regular weight) ----
GREEN  = "\033[0;32m"
DIM    = "\033[2;37m"
YELLOW = "\033[0;33m"
RED    = "\033[0;31m"
END    = "\033[0m"

def step(n, msg):
    print(f"{GREEN}[{n}/{TOTAL}] {msg} ...{END}", flush=True)

def ok(msg=""):
    print(f"{GREEN}       ✔ {msg}{END}", flush=True)

def warn(msg):
    print(f"{YELLOW}       ⚠ {msg}{END}", flush=True)

def fail(msg):
    print(f"{RED}       ✘ {msg}{END}", flush=True)

class _Greenify:
    def __init__(self, real): self.real = real
    def write(self, s):
        if not s: return self.real.write(s)
        out = []
        for line in s.splitlines(keepends=True):
            stripped = line.rstrip("\n\r")
            if stripped and "\033[" not in line:
                nl = line[len(stripped):]
                out.append(f"{GREEN}{stripped}{END}{nl}")
            else:
                out.append(line)
        return self.real.write("".join(out))
    def flush(self): return self.real.flush()
    def __getattr__(self, name): return getattr(self.real, name)

sys.stdout = _Greenify(sys.stdout)
sys.stderr = _Greenify(sys.stderr)

def run(cmd, label):
    t0 = time.time()
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    dt = time.time() - t0
    if r.returncode != 0:
        tail = (r.stderr.strip().splitlines() or ["error"])[-1]
        fail(f"{label} failed: {tail}")
        raise SystemExit(r.returncode)
    return dt

# ---- steps ----
t_start = time.time()
app_dir = Path.home() / ".promptgen"
app_dir.mkdir(parents=True, exist_ok=True)

step(1, "Mount Google Drive")
if Mount_Drive:
    from google.colab import drive
    drive.mount("/content/gdrive")
    ok("mounted at /content/gdrive")
else:
    warn("Mount_Drive is off, skipping")

step(2, "Copy client_secrets.json")
if not Client_Secrets_Path.strip():
    warn("no path provided, skipping")
elif not Mount_Drive:
    warn("Drive not mounted, skipping")
else:
    src = Path("/content/gdrive") / Client_Secrets_Path.strip().lstrip("/")
    dst = app_dir / "client_secrets.json"
    if src.exists():
        shutil.copy(src, dst)
        ok(f"copied to {dst}")
    else:
        warn(f"not found: {src}")

step(3, "Clone repository")
if os.path.exists(REPO_DIR):
    ok("already present, skipping clone")
else:
    dt = run(f"git clone -q {REPO_URL}", "git clone")
    ok(f"cloned in {dt:.1f}s")
os.chdir(REPO_DIR)

step(4, "Install dependencies")
dt = run("pip install -q .", "pip install")
ok(f"installed in {dt:.1f}s")

step(5, "Configure LoRA root folder")
if FOLDER_ID.strip():
    from src import config as cfgmod
    cfgmod.set_setting("loras_root_id", FOLDER_ID.strip())
    ok(f"loras_root_id = {FOLDER_ID.strip()}")
else:
    warn("FOLDER_ID empty, skipping")

total_dt = time.time() - t_start
print()
print(f"{DIM}setup done in {total_dt:.1f}s{END}")
print(f"{GREEN}Launching web interface ...{END}")
print()
from src import webapp
webapp.run(share=Public_Share, port=Port, inline=False)